In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

DATASET_PATH = r"donateacry_corpus"

OUTPUT_CSV = r"CSVS\base_audio.csv"

In [ ]:
class_mapping = {
    "discomfort": 0,
    "hungry": 1,
    "tired": 2
}

In [ ]:
files = []

for name, code in class_mapping.items():
    folder = os.path.join(DATASET_PATH, name)

    for file in os.listdir(folder):
        if file.endswith(".wav"):
            files.append({
                "Path": os.path.join(folder, file),
                "Class": code
            })

df = pd.DataFrame(files)
df = df.sample(frac=1, random_state=42)
df.to_csv(OUTPUT_CSV, index=False)

print("Total files:", len(df))
print(df["Class"].value_counts())

In [ ]:
df = pd.DataFrame(files, columns=["Path", "Class"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.to_csv(OUTPUT_CSV, index=False)

print(df.head())

In [ ]:
INPUT_CSV = r"CSVS\base_audio.csv"

TRAIN_CSV = r"CSVS\train.csv"

TEST_CSV = r"CSVS\test.csv"

In [ ]:
df = pd.read_csv(INPUT_CSV)

print("Original dataset:")
print(df["Class"].value_counts().sort_index())

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["Class"],
    random_state=42
)

In [ ]:
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df.to_csv(TRAIN_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)

In [ ]:
print("\nTRAIN SET:")
print(f"Total: {len(train_df)}")
print(train_df["Class"].value_counts().sort_index())

print("\nTEST SET:")
print(f"Total: {len(test_df)}")
print(test_df["Class"].value_counts().sort_index())


In [ ]:
import os
import shutil
import random

import pandas as pd
import librosa
import soundfile as sf
import numpy as np

TRAIN_CSV = r"CSVS\train.csv"
OUTPUT = r"edited_audio"

SR = 8000
TARGET = 450

if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)

os.makedirs(os.path.join(OUTPUT, "discomfort"))
os.makedirs(os.path.join(OUTPUT, "tired"))

df = pd.read_csv(TRAIN_CSV)

rng = np.random.default_rng(42)
random.seed(42)

def augment_audio(x):
    y = x.copy()

    choice = random.randint(0, 5)

    if choice == 0:
        y = librosa.effects.pitch_shift(
            y,
            sr=SR,
            n_steps=rng.uniform(-1.0, 1.0)
        )

    elif choice == 1:
        y = librosa.effects.time_stretch(
            y,
            rate=rng.uniform(0.97, 1.03)
        )

    elif choice == 2:
        gain = rng.uniform(0.75, 1.25)
        y = y * gain

    elif choice == 3:
        noise_level = rng.uniform(0.001, 0.008)
        noise = rng.normal(0, noise_level, len(y))
        y = y + noise

    elif choice == 4:
        shift = rng.uniform(-0.08, 0.08)
        y = y * (1 + shift)

    elif choice == 5:
        y = librosa.effects.pitch_shift(
            y,
            sr=SR,
            n_steps=rng.uniform(-0.7, 0.7)
        )
        y = librosa.effects.time_stretch(
            y,
            rate=rng.uniform(0.98, 1.02)
        )

    peak = np.max(np.abs(y))

    if peak > 1:
        y = y / peak

    return y

for code, folder in [(0, "discomfort"), (2, "tired")]:
    class_df = df[df["Class"] == code]

    files = []

    for _, row in class_df.iterrows():
        path = row["Path"]

        if not os.path.exists(path):
            continue

        try:
            x, _ = librosa.load(path, sr=SR, mono=True)

            if len(x) >= int(SR * 0.1):
                files.append((path, x))

        except Exception as e:
            print(f"Skipping {path}: {e}")

    original_count = len(files)
    needed = max(TARGET - original_count, 0)

    print(f"{folder}: {original_count} original files")
    print(f"{folder}: creating {needed} augmented files")

    if original_count == 0:
        print(f"No valid files found for {folder}")
        continue

    for i in range(needed):
        path, x = files[i % original_count]

        y = augment_audio(x)

        name = os.path.splitext(os.path.basename(path))[0]

        new_path = os.path.join(
            OUTPUT,
            folder,
            f"{name}_aug_{i}.wav"
        )

        sf.write(new_path, y, SR)

    print(f"{folder}: done")

print("Augmentation complete.")

In [ ]:
import os
import pandas as pd

TRAIN_CSV = r"CSVS\train.csv"
TEST_CSV = r"CSVS\test.csv"

paths = [r"ML\edited_audio", r"ML\donateacry_corpus"]
classes = {"discomfort": 0, "hungry": 1, "tired": 2}

train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

existing = set(train["Path"].str.lower())
existing.update(test["Path"].str.lower())

new_files = []

for path in paths:
    if not os.path.exists(path):
        continue

    for root, _, files in os.walk(path):
        name = os.path.basename(root).lower()

        if name not in classes:
            continue

        for file in files:
            if file.lower().endswith(".wav"):
                full_path = os.path.normpath(os.path.join(root, file))

                if full_path.lower() not in existing:
                    new_files.append({
                        "Path": full_path,
                        "Class": classes[name]
                    })
                    existing.add(full_path.lower())

if new_files:
    train = pd.concat(
        [train, pd.DataFrame(new_files)],
        ignore_index=True
    )
    train.to_csv(TRAIN_CSV, index=False)

print("Added:", len(new_files))
print(train["Class"].value_counts().sort_index())

In [ ]:
train_csv_df = pd.read_csv(r"CSVS\train.csv")

print("Train samples:")
print("Discomfort:", (train_csv_df["Class"] == 0).sum())
print("Hungry:", (train_csv_df["Class"] == 1).sum())
print("Tired:", (train_csv_df["Class"] == 2).sum())
print("Total:", len(train_csv_df))

In [ ]:
import os
import pandas as pd
import ML.feature_extraction as feature_extraction

FEATURE_COLUMNS = [
    "Amplitude_Envelope", "RMS", "ZCR", "STFT_Mean",
    "Spectral_Centroid", "Spectral_Bandwidth", "Spectral_Contrast",
    "Spectral_Rolloff", "F0",
    "MFCC_1", "MFCC_2", "MFCC_3", "MFCC_4", "MFCC_5",
    "MFCC_6", "MFCC_7", "MFCC_8", "MFCC_9", "MFCC_10",
    "MFCC_11", "MFCC_12", "MFCC_13", "Duration"
]

def create_features(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    rows = []

    for _, row in df.iterrows():
        path = row["Path"]

        if not os.path.isfile(path):
            continue

        values = feature_extraction.extract_features(path)

        data = {"Path": path, "Class": row["Class"]}
        data.update(dict(zip(FEATURE_COLUMNS, values)))
        rows.append(data)

    pd.DataFrame(rows).to_csv(output_csv, index=False)

create_features(r"CSVS\train.csv", r"CSVS\train_features.csv")
create_features(r"CSVS\test.csv", r"CSVS\test_features.csv")

In [ ]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix


TRAIN_FEATURES_CSV = r"CSVS\train_features.csv"
TEST_FEATURES_CSV = r"CSVS\test_features.csv"

In [ ]:
train_df = pd.read_csv(TRAIN_FEATURES_CSV)

test_df = pd.read_csv(TEST_FEATURES_CSV)


In [ ]:
X_train = train_df[FEATURE_COLUMNS]
y_train = train_df["Class"]


X_test = test_df[FEATURE_COLUMNS]
y_test = test_df["Class"]

In [ ]:
CLASS_WEIGHT = {0: 3.0, 1: 1, 2: 1.0}

model = RandomForestClassifier(
    n_estimators=500,
    class_weight=CLASS_WEIGHT,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Discomfort", "Hungry", "Tired"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
import joblib

joblib.dump(model, "cry_classifier2.joblib")
print("Model saved")

In [ ]:
print("done")